# Nested to Long and back

From https://stackoverflow.com/a/69604744

In [1]:
import pandas as pd
# import markdown


Create a data frame with two columns, the seconds of which has nested lists, sometimes with duplicated entries.

Notice that "apple" appears twice in the second column of the first row.

In [2]:
data = pd.DataFrame()
data['id'] = ["ab3e3", "psdds2", "pas13", "ccdf2", "dsda1"]
data['fruit'] = ["apple, orange, apple", "others", "dragon fruit, orange", "watermelon", "others"]

df = pd.DataFrame(data)
df


,id,fruit
0,ab3e3,"apple, orange, apple"
1,psdds2,others
2,pas13,"dragon fruit, orange"
3,ccdf2,watermelon
4,dsda1,others


## From Nested to Long using `.explode()`

Split elements in a column into a list.

In [3]:
df["fruit"].str.split(", ")


,fruit
0,"[apple, orange, apple]"
1,[others]
2,"[dragon fruit, orange]"
3,[watermelon]
4,[others]


Split elements in a column into a list and assign it to a new column in-line.

In [4]:
# the assign does not modify the data frame but creates a copy/view
df.assign(foobar=df["fruit"].str.split(", "))


,id,fruit,foobar
0,ab3e3,"apple, orange, apple","[apple, orange, apple]"
1,psdds2,others,[others]
2,pas13,"dragon fruit, orange","[dragon fruit, orange]"
3,ccdf2,watermelon,[watermelon]
4,dsda1,others,[others]


Notice that the data frame is unmodified.

In [5]:
df

,id,fruit
0,ab3e3,"apple, orange, apple"
1,psdds2,others
2,pas13,"dragon fruit, orange"
3,ccdf2,watermelon
4,dsda1,others


In [6]:
# similar to this ...
df_copy = df.copy()
df_copy['foobar'] = df_copy["fruit"].str.split(", ")
df_copy


,id,fruit,foobar
0,ab3e3,"apple, orange, apple","[apple, orange, apple]"
1,psdds2,others,[others]
2,pas13,"dragon fruit, orange","[dragon fruit, orange]"
3,ccdf2,watermelon,[watermelon]
4,dsda1,others,[others]


Notice that if use use an existing column name for the assign, the original column is replaced.

In [7]:
df.assign(fruit=df["fruit"].str.split(", "))


,id,fruit
0,ab3e3,"[apple, orange, apple]"
1,psdds2,[others]
2,pas13,"[dragon fruit, orange]"
3,ccdf2,[watermelon]
4,dsda1,[others]


In [8]:
df

,id,fruit
0,ab3e3,"apple, orange, apple"
1,psdds2,others
2,pas13,"dragon fruit, orange"
3,ccdf2,watermelon
4,dsda1,others


Explode the list into one row per element in the list.

In [9]:
df_explode = df.assign(fruit=df["fruit"].str.split(", ")).explode("fruit")
df_explode


,id,fruit
0,ab3e3,apple
0,ab3e3,orange
0,ab3e3,apple
1,psdds2,others
2,pas13,dragon fruit
2,pas13,orange
3,ccdf2,watermelon
4,dsda1,others


Notice that the row labels and the elements in the id column have been replicated.

## From Long to Nested using `.group_by()`

We can use the `.group_by()` method to reverse the process, recreating the nested structure as a string.

In [10]:
(
df_explode
  .groupby(['id'], as_index=False)['fruit']
  .agg(', '.join)
)


,id,fruit
0,ab3e3,"apple, orange, apple"
1,ccdf2,watermelon
2,dsda1,others
3,pas13,"dragon fruit, orange"
4,psdds2,others


Recreating the nested structure as a list.

In [11]:
(
df_explode
  .groupby('id', as_index=False)['fruit']
  .agg(list)
)


,id,fruit
0,ab3e3,"[apple, orange, apple]"
1,ccdf2,[watermelon]
2,dsda1,[others]
3,pas13,"[dragon fruit, orange]"
4,psdds2,[others]
